# 数据加载代码

In [7]:
import openml
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 你的核心加载函数（写得非常好！）
# =============================================================================
def load_dataset(name, task_type):
    """加载并预处理单个数据集"""
    
    # 数据集映射（100%匹配原任务）
    dataset_map = {
        # 分类数据集
        'diabetes': ('name', 'diabetes'),
        'credit-g': ('name', 'credit-g'),
        'MagicTelescope': ('task', 361065),
        'credit-default': ('task', 361055),
        'MiniBooNE': ('task', 361068),
        # 回归数据集
        'boston': ('name', 'boston'),
        'wine_quality': ('task', 361076),
        'superconduct': ('task', 361088),
        'fried': ('name', 'fried'),
        'diamonds': ('task', 361080),
    }
    
    lookup_type, lookup_value = dataset_map[name]
    
    if lookup_type == 'name':
        dataset = openml.datasets.get_dataset(lookup_value)
        X, y, _, _ = dataset.get_data(
            target=dataset.default_target_attribute,
            dataset_format='dataframe'
        )
    else:  # task
        task = openml.tasks.get_task(lookup_value)
        dataset = task.get_dataset()
        X, y, _, _ = dataset.get_data(
            target=task.target_name,
            dataset_format='dataframe'
        )
    
    # ✅ 坑1：清理特征名（LightGBM对$[]特殊字符敏感）
    X.columns = [c.replace('$', '_').replace('[', '_').replace(']', '_') 
                 for c in X.columns]
    
    # 处理分类特征
    categorical_cols = X.select_dtypes(include=['category', 'object']).columns
    for col in categorical_cols:
        X[col] = X[col].astype('category').cat.codes
    
    # ✅ 坑2：用SimpleImputer规范处理缺失值
    imputer = SimpleImputer(strategy='median')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    
    # 处理目标变量
    if task_type == 'classification':
        y = y.astype('category').cat.codes
        y = y.astype(int)
    
    # 划分数据集 80/20，固定随机种子42
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if task_type == 'classification' else None
    )
    
    # 特征标准化（深度学习需要，树模型不需要）
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # ✅ 坑3：回归目标标准化（TabNet必须，90%的人在这里翻车）
    y_scaler = None
    if task_type == 'regression':
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1)).flatten()
    else:
        y_train_scaled = y_train.values
        y_test_scaled = y_test.values
    
    return {
        'name': name,
        'X_train': X_train.values.astype(np.float32),
        'X_test': X_test.values.astype(np.float32),
        'X_train_scaled': X_train_scaled.astype(np.float32),
        'X_test_scaled': X_test_scaled.astype(np.float32),
        'y_train': y_train.values,
        'y_test': y_test.values,
        'y_train_scaled': y_train_scaled,
        'y_test_scaled': y_test_scaled,
        'y_scaler': y_scaler,
        'n_samples': len(X),
        'n_features': X.shape[1],
        'task_type': task_type
    }

# =============================================================================
# 批量加载所有10个数据集 + 验证
# =============================================================================
if __name__ == "__main__":
    print("="*60)
    print("开始加载原任务指定的10个数据集...")
    print("="*60)
    
    # 所有数据集配置
    ALL_DATASETS = [
        ('diabetes', 'classification'),
        ('credit-g', 'classification'),
        ('MagicTelescope', 'classification'),
        ('credit-default', 'classification'),
        ('MiniBooNE', 'classification'),
        ('boston', 'regression'),
        ('wine_quality', 'regression'),
        ('superconduct', 'regression'),
        ('fried', 'regression'),
        ('diamonds', 'regression'),
    ]
    
    all_datasets = {}
    for name, task_type in ALL_DATASETS:
        all_datasets[name] = load_dataset(name, task_type)
    
    print("="*60)
    print("✅ 所有数据集加载完成！")
    print("="*60)
    
    # 验证：和原任务表格100%匹配
    print("\n📊 与原任务对比验证:")
    print(f"{'数据集':20s} | {'类型':12s} | {'原任务样本':>10s} | {'实际样本':>10s} | {'匹配'}")
    print("-"*70)
    
    expected_samples = {
        'diabetes': 768, 'credit-g': 1000, 'MagicTelescope': 13376,
        'credit-default': 16714, 'MiniBooNE': 72998, 'boston': 506,
        'wine_quality': 6497, 'superconduct': 21263, 'fried': 40768, 'diamonds': 53940
    }
    
    for name, data in all_datasets.items():
        match = "✓" if data['n_samples'] == expected_samples[name] else "✗"
        print(f"{name:20s} | {data['task_type']:12s} | {expected_samples[name]:10d} | {data['n_samples']:10d} | {match}")
    
    print("\n💾 数据集已保存到 all_datasets 字典中，训练模型直接调用即可！")

开始加载原任务指定的10个数据集...
✅ 所有数据集加载完成！

📊 与原任务对比验证:
数据集                  | 类型           |      原任务样本 |       实际样本 | 匹配
----------------------------------------------------------------------
diabetes             | classification |        768 |        768 | ✓
credit-g             | classification |       1000 |       1000 | ✓
MagicTelescope       | classification |      13376 |      13376 | ✓
credit-default       | classification |      16714 |      16714 | ✓
MiniBooNE            | classification |      72998 |      72998 | ✓
boston               | regression   |        506 |        506 | ✓
wine_quality         | regression   |       6497 |       6497 | ✓
superconduct         | regression   |      21263 |      21263 | ✓
fried                | regression   |      40768 |      40768 | ✓
diamonds             | regression   |      53940 |      53940 | ✓

💾 数据集已保存到 all_datasets 字典中，训练模型直接调用即可！


8 个模型的完整实现代码
通用设置

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, r2_score
import time

# 固定所有随机种子
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### 树模型实现
Random Forest:

In [9]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

def train_random_forest(data):
    task_type = data['task_type']
    
    if task_type == 'classification':
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42,
            n_jobs=-1
        )
    else:
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42,
            n_jobs=-1
        )
    
    start_time = time.time()
    model.fit(data['X_train'], data['y_train'])
    train_time = time.time() - start_time
    
    y_pred = model.predict_proba(data['X_test'])[:, 1] if task_type == 'classification' \
        else model.predict(data['X_test'])
    
    score = roc_auc_score(data['y_test'], y_pred) if task_type == 'classification' \
        else r2_score(data['y_test'], y_pred)
    
    return {'score': score, 'train_time': train_time, 'model': model}

XGBoost:

In [10]:
from xgboost import XGBClassifier, XGBRegressor

def train_xgboost(data):
    task_type = data['task_type']
    
    if task_type == 'classification':
        model = XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            use_label_encoder=False,
            eval_metric='logloss'
        )
    else:
        model = XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
    
    start_time = time.time()
    model.fit(data['X_train'], data['y_train'])
    train_time = time.time() - start_time
    
    y_pred = model.predict_proba(data['X_test'])[:, 1] if task_type == 'classification' \
        else model.predict(data['X_test'])
    
    score = roc_auc_score(data['y_test'], y_pred) if task_type == 'classification' \
        else r2_score(data['y_test'], y_pred)
    
    return {'score': score, 'train_time': train_time, 'model': model}

LightGBM:

In [11]:
from lightgbm import LGBMClassifier, LGBMRegressor

def train_lightgbm(data):
    task_type = data['task_type']
    
    if task_type == 'classification':
        model = LGBMClassifier(
            n_estimators=100,
            max_depth=-1,
            learning_rate=0.1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
    else:
        model = LGBMRegressor(
            n_estimators=100,
            max_depth=-1,
            learning_rate=0.1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
    
    start_time = time.time()
    model.fit(data['X_train'], data['y_train'])
    train_time = time.time() - start_time
    
    y_pred = model.predict_proba(data['X_test'])[:, 1] if task_type == 'classification' \
        else model.predict(data['X_test'])
    
    score = roc_auc_score(data['y_test'], y_pred) if task_type == 'classification' \
        else r2_score(data['y_test'], y_pred)
    
    return {'score': score, 'train_time': train_time, 'model': model}

CatBoost

In [12]:
from catboost import CatBoostClassifier, CatBoostRegressor

def train_catboost(data):
    task_type = data['task_type']
    
    if task_type == 'classification':
        model = CatBoostClassifier(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            random_seed=42,
            verbose=0,
            thread_count=-1
        )
    else:
        model = CatBoostRegressor(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            random_seed=42,
            verbose=0,
            thread_count=-1
        )
    
    start_time = time.time()
    model.fit(data['X_train'], data['y_train'])
    train_time = time.time() - start_time
    
    y_pred = model.predict_proba(data['X_test'])[:, 1] if task_type == 'classification' \
        else model.predict(data['X_test'])
    
    score = roc_auc_score(data['y_test'], y_pred) if task_type == 'classification' \
        else r2_score(data['y_test'], y_pred)
    
    return {'score': score, 'train_time': train_time, 'model': model}

# 深度学习模型实现
通用训练函数：

In [11]:
def train_deep_model(model_class, data, model_params, lr=0.001):
    """通用深度学习模型训练函数"""
    task_type = data['task_type']
    input_dim = data['X_train_scaled'].shape[1]
    output_dim = 1 if task_type == 'regression' else len(np.unique(data['y_train']))
    
    # 小数据集调整batch size
    n_samples = len(data['X_train_scaled'])
    batch_size = min(256, max(16, n_samples // 4))
    
    # 准备数据
    X_train = torch.FloatTensor(data['X_train_scaled'])
    y_train = torch.FloatTensor(data['y_train_scaled']) if task_type == 'regression' \
        else torch.LongTensor(data['y_train_scaled'])
    
    X_test = torch.FloatTensor(data['X_test_scaled'])
    y_test = torch.FloatTensor(data['y_test_scaled']) if task_type == 'regression' \
        else torch.LongTensor(data['y_test_scaled'])
    
    # 训练/验证划分 (最后20%作为验证)
    val_size = int(0.2 * len(X_train))
    X_train, X_val = X_train[:-val_size], X_train[-val_size:]
    y_train, y_val = y_train[:-val_size], y_train[-val_size:]
    
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    # 初始化模型
    model = model_class(input_dim, output_dim, **model_params).to(device)
    
    # 损失函数和优化器
    criterion = nn.MSELoss() if task_type == 'regression' else nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )
    
    # 训练循环
    start_time = time.time()
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    
    for epoch in range(100):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            if task_type == 'regression':
                outputs = outputs.squeeze()
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        # 验证
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                if task_type == 'regression':
                    outputs = outputs.squeeze()
                val_loss += criterion(outputs, batch_y).item()
        
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= 10:
                break
    
    train_time = time.time() - start_time
    
    # 加载最佳模型
    model.load_state_dict(best_state)
    
    # 评估
    model.eval()
    with torch.no_grad():
        outputs = model(X_test.to(device))
        if task_type == 'regression':
            y_pred = outputs.squeeze().cpu().numpy()
            # 反标准化
            y_pred = data['y_scaler'].inverse_transform(y_pred.reshape(-1, 1)).flatten()
            score = r2_score(data['y_test'], y_pred)
        else:
            y_pred = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            score = roc_auc_score(data['y_test'], y_pred)
    
    return {'score': score, 'train_time': train_time, 'model': model}

MLP实现

In [12]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.2):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(128),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(64),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, x):
        return self.layers(x)

def train_mlp(data):
    return train_deep_model(MLP, data, {'dropout': 0.2}, lr=0.001)

Resnet实现

In [13]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()
    
    def forward(self, x):
        residual = x
        out = self.layers(x)
        out += residual
        return self.relu(out)

class ResNet(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.2):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.res_blocks = nn.Sequential(*[
            ResidualBlock(128, dropout) for _ in range(3)
        ])
        self.output_layer = nn.Linear(128, output_dim)
    
    def forward(self, x):
        x = self.input_layer(x)
        x = self.res_blocks(x)
        return self.output_layer(x)

def train_resnet(data):
    return train_deep_model(ResNet, data, {'dropout': 0.2}, lr=0.001)

FT-Transformer 实现：

In [22]:
class FeatureTokenizer(nn.Module):
    def __init__(self, n_features, d_model=64):
        super().__init__()
        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(n_features)
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
    
    def forward(self, x):
        # x: [batch_size, n_features]
        batch_size = x.shape[0]
        # 每个特征单独嵌入: [batch_size, n_features, d_model]
        features = []
        for i, embedding in enumerate(self.feature_embeddings):
            features.append(embedding(x[:, i:i+1]))
        x = torch.stack(features, dim=1)
        # 添加CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, d_ffn=256, dropout=0.2):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ffn),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ffn, d_model)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Self-attention
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        # FFN
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

class FTTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, d_model=64, n_heads=4, d_ffn=256, dropout=0.2):
        super().__init__()
        self.tokenizer = FeatureTokenizer(input_dim, d_model)
        self.transformers = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, d_ffn, dropout) for _ in range(3)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, output_dim)
    
    def forward(self, x):
        x = self.tokenizer(x)
        x = self.transformers(x)
        x = self.norm(x)
        # 使用CLS token输出
        cls_output = x[:, 0]
        return self.head(cls_output)

def train_ft_transformer(data):
    return train_deep_model(FTTransformer, data, {
        'd_model': 64, 'n_heads': 4, 'd_ffn': 256, 'dropout': 0.2
    }, lr=0.001)

TabNet 实现：

In [15]:
from pytorch_tabnet.tab_model import TabNetClassifier, TabNetRegressor

def train_tabnet(data):
    task_type = data['task_type']
    
    if task_type == 'classification':
        model = TabNetClassifier(
            n_d=4, n_a=4, n_steps=2,
            gamma=1.3,
            lambda_sparse=1e-4,
            seed=42,
            verbose=0
        )
    else:
        model = TabNetRegressor(
            n_d=4, n_a=4, n_steps=2,
            gamma=1.3,
            lambda_sparse=1e-4,
            seed=42,
            verbose=0
        )
    
    start_time = time.time()
    
    if task_type == 'classification':
        model.fit(
            data['X_train_scaled'], data['y_train'],
            max_epochs=100,
            patience=10,
            batch_size=256,
            virtual_batch_size=128,
            eval_set=[(data['X_test_scaled'], data['y_test'])],
        
        )
        y_pred = model.predict_proba(data['X_test_scaled'])[:, 1]
        score = roc_auc_score(data['y_test'], y_pred)
    else:
        # 注意：回归必须使用标准化的y！
        model.fit(
            data['X_train_scaled'], data['y_train_scaled'].reshape(-1, 1),
            max_epochs=100,
            patience=10,
            batch_size=256,
            virtual_batch_size=128,
            eval_set=[(data['X_test_scaled'], data['y_test_scaled'].reshape(-1, 1))],
           
        )
        y_pred_scaled = model.predict(data['X_test_scaled']).flatten()
        # 反标准化
        y_pred = data['y_scaler'].inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        score = r2_score(data['y_test'], y_pred)
    
    train_time = time.time() - start_time
    
    return {'score': score, 'train_time': train_time, 'model': model}

# 主实验脚本

In [23]:
def run_all_experiments():
    """运行所有实验"""
    
    # 所有数据集配置
    datasets = [
        # 分类数据集
        ('diabetes', 'classification'),
        ('credit-g', 'classification'),
        ('MagicTelescope', 'classification'),
        ('credit-default', 'classification'),
        ('MiniBooNE', 'classification'),
        # 回归数据集
        ('boston', 'regression'),
        ('wine_quality', 'regression'),
        ('superconduct', 'regression'),
        ('fried', 'regression'),
        ('diamonds', 'regression'),
    ]
    
    # 所有模型
    models = {
        # 树模型
        'RandomForest': train_random_forest,
        'XGBoost': train_xgboost,
        'LightGBM': train_lightgbm,
        'CatBoost': train_catboost,
        # 深度学习模型
        'MLP': train_mlp,
        'ResNet': train_resnet,
        'FT-Transformer': train_ft_transformer,
        'TabNet': train_tabnet,
    }
    
    results = {}
    
    for dataset_name, task_type in tqdm(datasets, desc='Datasets'):
        print(f"\n=== Processing {dataset_name} ({task_type}) ===")
        
        # 加载数据
        data = load_dataset(dataset_name, task_type)
        results[dataset_name] = {
            'task_type': task_type,
            'n_samples': data['n_samples'],
            'n_features': data['n_features'],
            'models': {}
        }
        
        for model_name, train_func in tqdm(models.items(), desc='Models', leave=False):
            print(f"  Training {model_name}...")
            try:
                result = train_func(data)
                results[dataset_name]['models'][model_name] = {
                    'score': float(result['score']),
                    'train_time': float(result['train_time']),
                    'model_type': 'tree-based' if model_name in ['RandomForest', 'XGBoost', 'LightGBM', 'CatBoost'] else 'deep-learning'
                }
                print(f"    Score: {result['score']:.4f}, Time: {result['train_time']:.2f}s")
            except Exception as e:
                print(f"    Error: {str(e)}")
                results[dataset_name]['models'][model_name] = {
                    'score': None,
                    'train_time': None,
                    'error': str(e)
                }
    
    # 保存结果
    import json
    with open('results/all_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    return results

# 运行所有实验
if __name__ == '__main__':
    import os
    os.makedirs('results', exist_ok=True)
    os.makedirs('figures', exist_ok=True)
    
    results = run_all_experiments()
    print("\n=== All experiments completed! ===")

Datasets:   0%|          | 0/10 [00:00<?, ?it/s]


=== Processing diabetes (classification) ===


  Training RandomForest...
    Score: 0.8118, Time: 0.16s


Models:  12%|█▎        | 1/8 [00:00<00:01,  5.13it/s]

  Training XGBoost...
    Score: 0.8200, Time: 0.04s
  Training LightGBM...
    Score: 0.8233, Time: 0.04s
  Training CatBoost...


...

    Score: 0.9394, Time: 25.12s
  Training ResNet...


    Score: 0.9447, Time: 23.70s
  Training FT-Transformer...


    Score: 0.9461, Time: 210.47s
  Training TabNet...

Early stopping occurred at epoch 28 with best_epoch = 18 and best_val_0_mse = 0.05582


Datasets: 100%|██████████| 10/10 [1:07:03<00:00, 402.31s/it]

    Score: 0.9443, Time: 51.92s

=== All experiments completed! ===


In [17]:
%pip install tqdm
from tqdm import tqdm

Note: you may need to restart the kernel to use updated packages.


In [24]:
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pandas as pd

# ===================== 固定绘图格式（不要改动）=====================
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['SimHei']  # 防止中文乱码，不需要可删除
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 10

# ===================== 读取实验结果json（仅读这一个文件）=====================
with open("results/all_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

# ===================== 整理成DataFrame =====================
rows = []
for dataset_name, data_info in results.items():
    n_samples = data_info["n_samples"]
    task_type = data_info["task_type"]
    for model_name, model_res in data_info["models"].items():
        rows.append({
            "dataset": dataset_name,
            "task_type": task_type,
            "n_samples": n_samples,
            "model": model_name,
            "model_type": model_res["model_type"],
            "score": model_res["score"]
        })
df = pd.DataFrame(rows)

# 数据集按照样本量 从小到大排序（硬性要求）
df = df.sort_values("n_samples", ascending=True).reset_index(drop=True)

# ===================== 严格配色：树模型绿色、深度学习橙红色 =====================
color_map = {
    # 树模型（绿色系）
    "RandomForest": "#2E7D32",
    "XGBoost": "#388E3C",
    "LightGBM": "#43A047",
    "CatBoost": "#4CAF50",
    # 深度学习（橙红色系）
    "MLP": "#E64A19",
    "ResNet": "#F57C00",
    "FT-Transformer": "#FF9800",
    "TabNet": "#FFB74D"
}

# ===================== 创建双子图 =====================
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 7))

# 子图1：分类任务
df_cls = df[df["task_type"] == "classification"]
sns.barplot(
    data=df_cls,
    x="dataset",
    y="score",
    hue="model",
    palette=color_map,
    ax=ax1,
    edgecolor="white",
    linewidth=0.6
)
ax1.set_title("Classification Tasks (Sorted by Dataset Size: Small → Large)", fontweight="bold", fontsize=12)
ax1.set_ylabel("ROC-AUC Score", fontsize=11)
ax1.set_xlabel("")
ax1.tick_params(axis="x", rotation=45)
ax1.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax1.set_ylim(0.5, 1.0)

# 子图2：回归任务
df_reg = df[df["task_type"] == "regression"]
sns.barplot(
    data=df_reg,
    x="dataset",
    y="score",
    hue="model",
    palette=color_map,
    ax=ax2,
    edgecolor="white",
    linewidth=0.6
)
ax2.set_title("Regression Tasks (Sorted by Dataset Size: Small → Large)", fontweight="bold", fontsize=12)
ax2.set_ylabel("R² Score", fontsize=11)
ax2.set_xlabel("")
ax2.tick_params(axis="x", rotation=45)
ax2.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax2.set_ylim(-0.1, 1.0)

plt.tight_layout()
plt.savefig("figures/figure1_model_comparison.png", dpi=300, bbox_inches="tight")
plt.close()
print("✅ Figure1 绘制完成，保存至：figures/figure1_model_comparison.png")


✅ Figure1 绘制完成，保存至：figures/figure1_model_comparison.png


In [25]:
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pandas as pd
import numpy as np

# ===================== 固定绘图格式（和Figure1保持一致）=====================
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 10

# ===================== 读取实验结果json =====================
with open("results/all_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

# ===================== 整理成DataFrame =====================
rows = []
for dataset_name, data_info in results.items():
    task_type = data_info["task_type"]
    for model_name, model_res in data_info["models"].items():
        rows.append({
            "dataset": dataset_name,
            "task_type": task_type,
            "model": model_name,
            "model_type": model_res["model_type"],
            "score": model_res["score"]
        })
df = pd.DataFrame(rows)

# ===================== 计算每个数据集内排名（分数越高排名越靠前）=====================
def get_rank(group):
    group["rank"] = group["score"].rank(ascending=False, method="min")
    return group
df = df.groupby(["dataset","task_type"]).apply(get_rank).reset_index(drop=True)

# ===================== 计算平均排名 =====================
avg_rank = df.groupby(["model","model_type"])["rank"].agg(["mean","std"]).round(2)
avg_rank = avg_rank.sort_values("mean", ascending=True).reset_index()

# ===================== 沿用Figure1同款配色 =====================
color_map = {
    # 树模型（绿色系）
    "RandomForest": "#2E7D32",
    "XGBoost": "#388E3C",
    "LightGBM": "#43A047",
    "CatBoost": "#4CAF50",
    # 深度学习（橙红色系）
    "MLP": "#E64A19",
    "ResNet": "#F57C00",
    "FT-Transformer": "#FF9800",
    "TabNet": "#FFB74D"
}
# 匹配排序后的颜色
bar_color = [color_map[m] for m in avg_rank["model"]]

# ===================== 绘制Figure2 平均排名柱状图 =====================
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(avg_rank["model"], avg_rank["mean"], color=bar_color, edgecolor="white", linewidth=0.8)

# Y轴固定范围 1~8（作业硬性要求）
ax.set_ylim(1, 8)
ax.invert_yaxis()  # 排名1在最上方，可视化更直观

# 标题、坐标轴
ax.set_title("Average Ranking of Models Across All Datasets", fontweight="bold", fontsize=13)
ax.set_ylabel("Mean Rank (1 = Best, 8 = Worst)", fontsize=11)
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)

# ✅ 柱子标注精确排名数值（作业要求annotated with exact rank values）
for bar, rank_val in zip(bars, avg_rank["mean"]):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.15,
            f"{rank_val}", ha="center", fontsize=9, fontweight="bold")

# 辅助线：区分排名层级
ax.axhline(y=4.5, color="gray", linestyle="--", alpha=0.5)
ax.text(7, 5.0, "Mid Ranking Boundary", color="gray", fontsize=8)

plt.tight_layout()
plt.savefig("figures/figure2_average_ranking.png", dpi=300, bbox_inches="tight")
plt.close()
print("✅ Figure2 绘制完成，保存至：figures/figure2_average_ranking.png")


✅ Figure2 绘制完成，保存至：figures/figure2_average_ranking.png


In [27]:
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pandas as pd
import numpy as np

# ===================== 固定绘图格式（和Figure1/2完全统一、无报错）=====================
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']  # 英文通用字体，无中文依赖
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 10

# ===================== 读取实验结果json =====================
with open("results/all_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

# ===================== 整理数据 =====================
rows = []
for dataset_name, data_info in results.items():
    n_samples = data_info["n_samples"]
    task_type = data_info["task_type"]
    for model_name, model_res in data_info["models"].items():
        rows.append({
            "dataset": dataset_name,
            "task_type": task_type,
            "n_samples": n_samples,
            "model": model_name,
            "model_type": model_res["model_type"],
            "score": model_res["score"]
        })
df = pd.DataFrame(rows)

# ===================== 计算：每个数据集 树模型平均分 / 深度学习平均分 =====================
grouped = df.groupby(["dataset","task_type","n_samples","model_type"])["score"].mean().unstack()
grouped["gap"] = grouped["tree-based"] - grouped["deep-learning"]
grouped = grouped.reset_index()

# ===================== 绘图配置 =====================
plt.figure(figsize=(12, 7))

# 分类、回归使用不同标记（题目硬性要求）
marker_map = {"classification":"o", "regression":"^"}
color_map = {"classification":"#2E7D32", "regression":"#E64A19"}

# 绘制散点
for task in grouped["task_type"].unique():
    sub = grouped[grouped["task_type"] == task]
    plt.scatter(
        sub["n_samples"],
        sub["gap"],
        s=110,
        marker=marker_map[task],
        c=color_map[task],
        edgecolor="white",
        linewidth=1.2,
        label=f"{task.capitalize()}"
    )

# ✅ X轴对数刻度 log scale（题目要求）
plt.xscale("log")

# ✅ 黑色虚线 gap=0（题目要求）
plt.axhline(y=0, linestyle="--", color="black", alpha=0.7, label="Gap = 0 (Equal Performance)")

# ✅ 每个点标注数据集名称（题目要求labeled with dataset name）
for idx, row in grouped.iterrows():
    plt.text(
        row["n_samples"] * 1.08,
        row["gap"],
        row["dataset"],
        fontsize=8,
        ha="left"
    )

# 坐标轴与标题
plt.title("Tree-vs-Deep Performance Gap vs Dataset Size", fontweight="bold", fontsize=13)
plt.xlabel("Dataset Size (Log Scale)", fontsize=11)
plt.ylabel("Average Tree Score − Average Deep Score", fontsize=11)

# 图例
plt.legend(loc="best")
plt.tight_layout()

# 保存图片
plt.savefig("figures/figure3_gap_scatter.png", dpi=300, bbox_inches="tight")
plt.close()
print("✅ Figure3 绘制完成，保存至：figures/figure3_gap_scatter.png")




✅ Figure3 绘制完成，保存至：figures/figure3_gap_scatter.png
